
#  Priors 


In [1]:
# --- Output folder override
from pathlib import Path
FIG_DIR = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] Figures will be saved to: {FIG_DIR}")


[setup] Figures will be saved to: C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figures


In [ ]:

# --- Config (edit as needed) ---
PRED_Q_LO = 0.10      # predictive lower quantile (fraction, e.g., 0.10)
PRED_Q_HI = 0.90      # predictive upper quantile (fraction, e.g., 0.90)
ROLL_WINDOW = None    # e.g., '7D' to show a rolling mean of AF (optional)
MAX_SITES = None      # limit plotting to first N sites (None = all)
MAX_MUTS  = None      # limit plotting to first N mutations overall (None = all)
MAX_PAIRS = None      # limit (site,mutation) detail pages (None = all)
SAVE_PNG  = True
SAVE_PDF  = True
DPI       = 150
PAGESIZE  = (12, 8)   # width, height in inches for figures

# --- Auto-detect repo root ---
import os
from pathlib import Path

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    # Walk up looking for 'results/priors/priors_full_detail.csv' or 'results' / 'configs'
    while True:
        if (p/'results'/'priors'/'priors_full_detail.csv').exists():
            return p
        if (p/'.git').exists() or (p/'configs').exists():
            root_guess = p
        if p.parent == p:
            break
        p = p.parent
    # Fallback: current working directory
    return Path(start or os.getcwd()).resolve()

REPO_ROOT = find_repo_root()
PRIOR_DIR = REPO_ROOT / 'results' / 'priors'
FIG_DIR   = FIG_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"[info] REPO_ROOT = {REPO_ROOT}")
print(f"[info] Using results in {PRIOR_DIR}")
print(f"[info] Figures -> {FIG_DIR}")

# --- Imports & Styles (no-TeX) ---
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator

import seaborn as sns
import scienceplots  # style only
import colorcet as cc
import cmocean
from palettable.colorbrewer.qualitative import Set3_12
from palettable.cartocolors.qualitative import Safe_10

# Use SciencePlots *without LaTeX*
plt.style.use(['science', 'no-latex', 'bright'])

# Nice defaults; no TeX anywhere.
mpl.rcParams.update({
    "axes.unicode_minus": False,
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
    "savefig.bbox": "tight",
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

sns.set_theme(style="whitegrid", context="notebook")
# Distinct categorical palette for many series
sns.set_palette(sns.color_palette(cc.glasbey[:20]))

plt.ioff()  # do not try to display figures inline while saving lots of files


[info] REPO_ROOT = C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting
[info] Using results in C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors
[info] Figures -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figures


In [ ]:

# ---- Required ----
p_full = PRIOR_DIR / 'priors_full_detail.csv'
assert p_full.exists(), f"Required file not found: {p_full}"

# Windows-safe filename sanitizer
import re
_INVALID = re.compile(r'[<>:"/\\|?*\x00-\x1F]')
def safe_fname(x: str, maxlen: int = 200) -> str:
    s = str(x)
    s = _INVALID.sub("_", s)
    s = s.replace(" ", "_").rstrip(". ").strip()
    return s[:maxlen] if maxlen else s

df = pd.read_csv(p_full, parse_dates=['date'])
# Normalize column names
df.rename(columns={'kappa_use':'kappa_t'}, inplace=True)
if 'af' not in df.columns:
    df['af'] = np.where(df['coverage']>0, df['count']/df['coverage'].clip(lower=1), np.nan)

# ---- Optionals ----
def _read_csv_opt(name, parse_dates=None):
    path = PRIOR_DIR / name
    return pd.read_csv(path, parse_dates=parse_dates) if path.exists() else None

hyper = _read_csv_opt('priors_hyperparams.csv')
gts   = _read_csv_opt('detail_global_timeseries.csv', parse_dates=['date'])
shifts= _read_csv_opt('time_shifts.csv')
eb    = _read_csv_opt('eb_population_prior.csv')
proc  = _read_csv_opt('process_noise_by_mutation.csv')

print('[info] Loaded: priors_full_detail.csv')
print(f"[info] Also found: "
      f"hyper={hyper is not None}, global_ts={gts is not None}, time_shifts={shifts is not None}, "
      f"eb={eb is not None}, process_noise={proc is not None}")


[info] Loaded: priors_full_detail.csv
[info] Also found: hyper=True, global_ts=True, time_shifts=False, eb=True, process_noise=False


In [ ]:

from scipy.stats import betabinom, kstest

from scipy.special import betainc, betaincinv, betaln
import numpy as np

def predictive_betabinom_bounds_fast(n, mu, kappa, q_lo=0.10, q_hi=0.90):
    n = np.asarray(n, int)
    mu = np.clip(np.asarray(mu, float), 1e-9, 1 - 1e-9)
    k = np.clip(np.asarray(kappa, float), 1e-8, 1e9)
    a, b = mu * k, (1 - mu) * k

    mask = n > 0
    lo = np.full_like(mu, np.nan, dtype=float)
    hi = np.full_like(mu, np.nan, dtype=float)

    if np.any(mask):
        # Use betaincinv on the regularized incomplete beta
        p_lo = betaincinv(a[mask], b[mask], q_lo)
        p_hi = betaincinv(a[mask], b[mask], q_hi)
        lo[mask] = np.floor(n[mask] * p_lo)
        hi[mask] = np.ceil(n[mask] * p_hi)
    return lo, hi


def midp_pit(y, n, mu, k):
    y = np.asarray(y, int); n = np.asarray(n, int)
    mu = np.clip(np.asarray(mu, float), 1e-9, 1-1e-9)
    k  = np.clip(np.asarray(k, float), 1e-8, 1e12)
    a, b = mu*k, (1-mu)*k
    Fy  = betabinom.cdf(y, n, a, b)
    pmf = betabinom.pmf(y, n, a, b)
    u = np.clip(Fy - 0.5*pmf, 0, 1)
    u[~np.isfinite(u)] = 0.5
    return u

lo_ct, hi_ct = predictive_betabinom_bounds(df['coverage'].to_numpy(),
                                           df['mu_t'].to_numpy(), df['kappa_t'].to_numpy(),
                                           q_lo=PRED_Q_LO, q_hi=PRED_Q_HI)
df['pred_lo_ct'] = lo_ct
df['pred_hi_ct'] = hi_ct
df['pred_lo_af'] = lo_ct / df['coverage'].replace(0, np.nan)
df['pred_hi_af'] = hi_ct / df['coverage'].replace(0, np.nan)

df['pit_mid'] = midp_pit(df['count'].to_numpy(),
                         df['coverage'].to_numpy(),
                         df['mu_t'].to_numpy(),
                         df['kappa_t'].to_numpy())
df['outlier'] = (df['af'] < df['pred_lo_af']) | (df['af'] > df['pred_hi_af'])
df['outlier'] = df['outlier'].fillna(False)

print('[info] Derived predictive bands and PIT in-memory.')


In [ ]:

# Apply optional limits to cut down plots if desired
sites_all = df['site_id'].dropna().astype(str).unique().tolist()
muts_all  = df['mutation'].dropna().astype(str).unique().tolist()

sites_sel = sites_all[:MAX_SITES] if isinstance(MAX_SITES, int) else sites_all
muts_sel  = muts_all[:MAX_MUTS]   if isinstance(MAX_MUTS, int)   else muts_all

df_sel = df[df['site_id'].isin(sites_sel) & df['mutation'].isin(muts_sel)].copy()

print(f"[info] Sites: {len(sites_sel)} of {len(sites_all)}; Mutations: {len(muts_sel)} of {len(muts_all)}")


In [ ]:

from matplotlib.gridspec import GridSpec

def _ax_format_time(ax, title=None):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6, prune=None))
    ax.set_xlabel('Date')
    ax.set_ylabel('Allele frequency')
    if title:
        ax.set_title(title, fontsize=11)

def plot_series_panel(ax, d, title=None):
    # shaded predictive band (AF)
    ax.fill_between(d['date'], d['pred_lo_af'], d['pred_hi_af'], alpha=0.2, label=f'Pred {int((PRED_Q_HI-PRED_Q_LO)*100)}% AF')
    # posterior median mu(t)
    ax.plot(d['date'], d['mu_t'], linewidth=2, label='μ(t)')
    # optional CI for μ, if present
    if {'mu_lo','mu_hi'}.issubset(d.columns):
        ax.fill_between(d['date'], d['mu_lo'], d['mu_hi'], alpha=0.15, label='μ 95% CI')
    # observations
    sz = np.clip(np.sqrt(d['coverage'].fillna(0))/4.0, 3, 16)  # marker size ~ sqrt(coverage)
    ax.scatter(d['date'], d['af'], s=sz, alpha=0.65, label='Observed AF')
    # outliers
    out = d[d['outlier']]
    if not out.empty:
        ax.scatter(out['date'], out['af'], s=sz[out.index], facecolors='none', edgecolors='r', linewidths=1.2, label='Outlier')
    _ax_format_time(ax, title=title)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc='best', fontsize=8, frameon=False)

def plot_pair_detail_page(fig, d, site_id, mutation):
    # page layout: 2x2
    gs = GridSpec(2, 2, figure=fig)
    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    # top: time series
    plot_series_panel(ax1, d, title=f"{site_id} — {mutation} (series)")
    # bottom-left: counts vs predictive band
    ax2.fill_between(d['date'], d['pred_lo_ct'], d['pred_hi_ct'], alpha=0.2)
    ax2.plot(d['date'], d['count'], linewidth=1.5)
    ax2.set_title('Counts vs Pred band')
    ax2.set_ylabel('Count')
    ax2.xaxis.set_major_locator(MaxNLocator(nbins=6))
    # bottom-right: PIT histogram
    u = d['pit_mid'].dropna().to_numpy()
    ax3.hist(u, bins=20, range=(0,1), density=True)
    ax3.set_title('PIT (mid-P)')
    ax3.set_xlabel('u')
    ax3.set_ylabel('density')


In [ ]:

# ---------- Sites dashboard ----------
site_pages = FIG_DIR / 'sites'
site_pages.mkdir(parents=True, exist_ok=True)

pdf_sites_path = FIG_DIR / 'sites_dashboard.pdf' if SAVE_PDF else None
pdf_sites = PdfPages(pdf_sites_path) if SAVE_PDF else None

for s in sites_sel:
    dsite = df_sel[df_sel['site_id'] == s]
    muts = sorted(dsite['mutation'].unique().tolist())
    if not muts:
        continue

    ncols = 3
    nrows = int(np.ceil(len(muts) / ncols))

    fig = plt.figure(figsize=(PAGESIZE[0], max(PAGESIZE[1], 3 + 3 * nrows)))
    gs = GridSpec(nrows, ncols, figure=fig)

    for i, m in enumerate(muts):
        r, c = divmod(i, ncols)
        ax = fig.add_subplot(gs[r, c])
        d = dsite[dsite['mutation'] == m].sort_values('date')
        plot_series_panel(ax, d, title=str(m))

    fig.suptitle(f"Site: {s}", fontsize=14, y=0.995)
    fig.tight_layout(rect=[0, 0.01, 1, 0.97])

    if SAVE_PNG:
        out = site_pages / f"site_{safe_fname(s)}.png"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")

    if pdf_sites is not None:
        pdf_sites.savefig(fig)

    plt.close(fig)

if pdf_sites is not None:
    pdf_sites.close()
    print(f"[info] Wrote {pdf_sites_path}")

print(f"[info] Per-site PNGs -> {site_pages}")


In [ ]:

# ---------- Mutations dashboard ----------
mut_pages = FIG_DIR / 'mutations'
mut_pages.mkdir(parents=True, exist_ok=True)

pdf_muts_path = FIG_DIR / 'mutations_dashboard.pdf' if SAVE_PDF else None
pdf_muts = PdfPages(pdf_muts_path) if SAVE_PDF else None

for m in muts_sel:
    dmut = df_sel[df_sel['mutation']==m]
    sites = sorted(dmut['site_id'].unique().tolist())
    if not sites:
        continue
    ncols = 3
    nrows = int(np.ceil(len(sites) / ncols))
    fig = plt.figure(figsize=(PAGESIZE[0], max(PAGESIZE[1], 3 + 3*nrows)))
    gs = GridSpec(nrows, ncols, figure=fig)
    for i, s in enumerate(sites):
        r, c = divmod(i, ncols)
        ax = fig.add_subplot(gs[r, c])
        d = dmut[dmut['site_id']==s].sort_values('date')
        plot_series_panel(ax, d, title=str(s))
    fig.suptitle(f"Mutation: {m}", fontsize=14, y=0.995)
    fig.tight_layout(rect=[0, 0.01, 1, 0.97])

    if SAVE_PNG:
        out = mut_pages / f"mutation_{safe_fname(m)}.png"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")

    if pdf_muts is not None:
        pdf_muts.savefig(fig)

    plt.close(fig)

if pdf_muts is not None:
    pdf_muts.close()
    print(f"[info] Wrote {pdf_muts_path}")

print(f"[info] Per-mutation PNGs -> {mut_pages}")


In [ ]:

# ---------- Pair detail pages ----------
pair_pages = FIG_DIR / 'pairs'
pair_pages.mkdir(parents=True, exist_ok=True)

pairs = (df_sel[['site_id','mutation']]
         .drop_duplicates()
         .sort_values(['site_id','mutation'])
         .to_records(index=False))

if isinstance(MAX_PAIRS, int):
    pairs = pairs[:MAX_PAIRS]

pdf_pairs_path = FIG_DIR / 'pair_details.pdf' if SAVE_PDF else None
pdf_pairs = PdfPages(pdf_pairs_path) if SAVE_PDF else None

for s, m in pairs:
    d = df_sel[(df_sel['site_id']==s) & (df_sel['mutation']==m)].sort_values('date')
    if d.empty:
        continue
    fig = plt.figure(figsize=(PAGESIZE[0], PAGESIZE[1]))
    plot_pair_detail_page(fig, d, s, m)
    fig.tight_layout()

    if SAVE_PNG:
        out = pair_pages / f"pair_{safe_fname(s)}__{safe_fname(m)}.png"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
    if pdf_pairs is not None:
        pdf_pairs.savefig(fig)
    plt.close(fig)

if pdf_pairs is not None:
    pdf_pairs.close()
    print(f"[info] Wrote {pdf_pairs_path}")

print(f"[info] Pair PNGs -> {pair_pages}")


In [ ]:

# ---------- Global summaries ----------
glob_dir = FIG_DIR / 'global'
glob_dir.mkdir(parents=True, exist_ok=True)

# Distribution plots: mu_t and kappa_t across all rows
fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(df['mu_t'].dropna(), bins=50, stat="density", ax=ax)
ax.set_title('Distribution of μ(t) across rows')
ax.set_xlabel('μ')
ax.set_ylabel('density')
if SAVE_PNG: fig.savefig(glob_dir / 'mu_distribution.png', dpi=DPI); plt.close(fig)

fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(df['kappa_t'].dropna(), bins=50, stat="density", ax=ax)
ax.set_title('Distribution of κ across rows')
ax.set_xlabel('κ')
ax.set_ylabel('density')
if SAVE_PNG: fig.savefig(glob_dir / 'kappa_distribution.png', dpi=DPI); plt.close(fig)

# Outlier rate by mutation
out_rate = (df.groupby('mutation')['outlier'].mean()
            .sort_values(ascending=False)
            .reset_index(name='outlier_rate'))
fig, ax = plt.subplots(figsize=(min(14, 4 + 0.2*len(out_rate)), 4))
sns.barplot(data=out_rate, x='mutation', y='outlier_rate', ax=ax)
ax.set_title('Outlier rate by mutation')
ax.set_ylabel('rate')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=60)
if SAVE_PNG: fig.savefig(glob_dir / 'outlier_rate_by_mutation.png', dpi=DPI); plt.close(fig)

# Rows per day
rows_per_date = df.groupby('date').size().reset_index(name='n')
fig, ax = plt.subplots(figsize=(12,4))
ax.bar(rows_per_date['date'], rows_per_date['n'], width=1.0)
ax.set_title('Number of rows per day')
ax.set_ylabel('rows')
ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
if SAVE_PNG: fig.savefig(glob_dir / 'rows_per_day.png', dpi=DPI); plt.close(fig)

# PIT histogram overall + KS test against Uniform(0,1)
u = df['pit_mid'].dropna().to_numpy()
ks_p = float('nan')
if u.size > 0:
    ks_p = kstest(u, 'uniform').pvalue
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(u, bins=30, stat="density", binrange=(0,1), ax=ax)
ax.set_title(f'Global PIT (mid-P) — KS p={ks_p:.3g}')
ax.set_xlabel('u')
ax.set_ylabel('density')
if SAVE_PNG: fig.savefig(glob_dir / 'pit_hist_global.png', dpi=DPI); plt.close(fig)

print(f"[info] Global summaries -> {glob_dir}")


In [ ]:

# ---------- Global μ(t) per mutation ----------
gmt_dir = FIG_DIR / 'global_ts'
gmt_dir.mkdir(parents=True, exist_ok=True)

if gts is not None and {'date','mutation','mu_t'}.issubset(gts.columns):
    gsrc = gts.copy()
else:
    # Fallback: unweighted mean μ per date, per mutation computed from priors_full_detail
    gsrc = (df.groupby(['date','mutation'], as_index=False)
              .agg(mu_t=('mu_t','mean')))

for m in muts_sel:
    d = gsrc[gsrc['mutation']==m].sort_values('date')
    if d.empty:
        continue
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(d['date'], d['mu_t'], linewidth=2)
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(f'Global μ(t) — {m}')
    ax.set_xlabel('Date'); ax.set_ylabel('μ')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    if SAVE_PNG:
        out = gmt_dir / f'global_mu_{safe_fname(m)}.png'
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

print(f"[info] Global μ(t) lines -> {gmt_dir}")


In [ ]:

# ---------- Coverage diagnostics by coverage decile ----------
cov = df[['coverage','count','pred_lo_ct','pred_hi_ct']].dropna()
cov = cov[cov['coverage']>0].copy()
if not cov.empty:
    cov['tile'] = pd.qcut(cov['coverage'], q=10, duplicates='drop')
    cov['covered'] = (cov['count'] >= cov['pred_lo_ct']) & (cov['count'] <= cov['pred_hi_ct'])
    cov_rate = cov.groupby('tile')['covered'].mean().reset_index(name='coverage_rate')

    fig, ax = plt.subplots(figsize=(10,4))
    sns.barplot(data=cov_rate, x='tile', y='coverage_rate', ax=ax)
    ax.set_title(f'Predictive coverage (counts) by coverage decile — target {(PRED_Q_HI-PRED_Q_LO):.0%}')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    if SAVE_PNG: fig.savefig(FIG_DIR / 'coverage_by_coverage_decile.png', dpi=DPI); plt.close(fig)


In [ ]:

# ---------- PIT KS by site ----------
ks_rows = []
for s in sites_sel:
    u = df_sel.loc[df_sel['site_id']==s, 'pit_mid'].dropna().to_numpy()
    if u.size >= 20:
        ks_p = kstest(u, 'uniform').pvalue
        ks_rows.append({'site_id': s, 'ks_p': ks_p, 'n': int(u.size)})
ks_df = pd.DataFrame(ks_rows).sort_values('ks_p') if ks_rows else pd.DataFrame(columns=['site_id','ks_p','n'])

if not ks_df.empty:
    fig, ax = plt.subplots(figsize=(min(14, 4 + 0.15*len(ks_df)), 5))
    sns.barplot(data=ks_df, x='site_id', y='ks_p', ax=ax)
    ax.set_title('PIT KS p-value by site (higher is better)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=60)
    if SAVE_PNG: fig.savefig(FIG_DIR / 'pit_ks_by_site.png', dpi=DPI); plt.close(fig)


In [ ]:

# ---------- κ vs μ scatter ----------
samp = df[['mu_t','kappa_t']].dropna()
if len(samp) > 200000:
    samp = samp.sample(200000, random_state=123)

fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(samp['mu_t'], samp['kappa_t'], s=6, alpha=0.3)
ax.set_xlabel('μ'); ax.set_ylabel('κ')
ax.set_title('Scatter of κ vs μ')
if SAVE_PNG: fig.savefig(FIG_DIR / 'scatter_kappa_vs_mu.png', dpi=DPI); plt.close(fig)


In [ ]:

# ---------- Process noise scatter ) ----------
if proc is not None and {'q_LL_hat','q_b_hat'}.issubset(proc.columns):
    x = np.clip(proc['q_LL_hat'].to_numpy(dtype=float), 1e-12, None)
    y = np.clip(proc['q_b_hat'].to_numpy(dtype=float), 1e-12, None)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(x, y, s=16, alpha=0.7)
    # Only log-scale if strictly positive
    if np.all(x > 0) and np.all(y > 0):
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('q_LL_hat'); ax.set_ylabel('q_b_hat')
    ax.set_title('Per-mutation process noise')
    if SAVE_PNG: fig.savefig(FIG_DIR / 'process_noise_scatter.png', dpi=DPI); plt.close(fig)

    # Marginal histograms (log10)
    fig, ax = plt.subplots(figsize=(6,4))
    ax.hist(np.log10(x), bins=40, alpha=0.9)
    ax.set_xlabel('log10 q_LL_hat'); ax.set_ylabel('Count'); ax.set_title('Distribution of q_LL_hat')
    if SAVE_PNG: fig.savefig(FIG_DIR / 'q_LL_hist_log10.png', dpi=DPI); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6,4))
    ax.hist(np.log10(y), bins=40, alpha=0.9)
    ax.set_xlabel('log10 q_b_hat'); ax.set_ylabel('Count'); ax.set_title('Distribution of q_b_hat')
    if SAVE_PNG: fig.savefig(FIG_DIR / 'q_b_hist_log10.png', dpi=DPI); plt.close(fig)
